# Box model using ATom data - temperature and pressure dependent, but with the errors in calculations for k so model is overpredicted

---

## ATom data

- This notebook makes use of measurements from deployment 1 of NASA's Atmospheric Tomography Mission (ATom), which was carried out in June-August 2016 and consisted of regular 0.2–12 km profiling. (https://doi.org/10.5194/essd-15-3299-2023)
- In this notebook we will look at a dataset containing mixing ratios of hydrogen oxides measured by the Airborne Tropospheric Hydrogen Oxides Sensor (ATHOS) from this deployment, and use it to build up an atmospheric 1-box, 0-dimensional model of OH concentrations. (https://doi.org/10.3334/ORNLDAAC/1877)
- https://jpldataeval.jpl.nasa.gov/pdf/NASA-JPL%20Evaluation%2019-5.pdf

---

## Inspect dataset

In [ ]:
import pandas as pd
df = pd.read_csv("MDS_atom1_2016_summer_with_no_no2_oh_ho2_full_js.csv")

In [ ]:
df.shape

In [ ]:
df.head()

Filtering out the dataset to only keep the columns of interest:

In [ ]:
df = df[['Temp', 'Pres','UTC_Start_dt', 'lat', 'lon', 'Altitude', 'jO3_O2_O1D_CAFS', 'O3_M', 'H2O_M', 'OH_ATHOS', 'CO_M', 'CH4_M', 'PAN_M']]
df

In [ ]:
from datetime import datetime, timezone
import numpy as np
from pytz import timezone
df["UTC_Start_dt"] = pd.to_datetime(df["UTC_Start_dt"])
df['UTC_Start_dt'] = df['UTC_Start_dt'].dt.tz_localize('UTC')
df["UTC_Start_dt"] = df["UTC_Start_dt"].dt.tz_convert('America/New_York')
df['date'] = df['UTC_Start_dt']
df['date']

In [ ]:
df_resampled = df.resample('H', on = 'UTC_Start_dt').mean()
df_resampled

In [ ]:
## Plot of altitude for one day
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["Altitude"][df["date"]< "2016-07-30"])
plt.xlabel("date")
plt.ylabel("altitude / m")
plt.title("date vs altitude")
plt.show()

## Source rate

Source of $OH$:

$O(^1D) + H_2O → 2OH$

$O(^1D) + M → O^3P + M$

Data used based on $O(^1D) + N_2 → O(^3P) + N$

The rate equation for this is $2 \cdot k[O^1D][H_2O]$

Need to calculate $[O^1D]$ indirectly using data for $[O_3]$

Assumed to be in steady state - due to its short lifetime

So $\frac{d[O^1D]}{dt} = J[O_3] - k_1[O^1D][H_2O] - k_2[O^1D][M] = 0$

$[O^1D]_{SS} = \frac{J[O_3]}{k_1[H_2O] + k_2[M]}$

$(M = \frac{p}{kT})$

$k = Ae^{-\frac{E_a}{RT}}$ to calculate temperature dependent rate constants

## Calculations for source rate
For rate constants - temperature dependent so use Arrhenius equation: $k = Ae^{-\frac{E_a}{RT}}$

Convert units for species so that everything is in units of $molecules \: cm^{-3}$ and calculate $[M]$ using $[M] = \frac{p}{kT}$ ($p$ is pressure in $Pa$, $k$ is Boltzmann's constant in $J \: K^-1$ and $T$ is temperature in $K$

$1 J = 1 \: kg \: m^2 \: s^{-2}$

$1 \: Pa = 1 \: N m^{-2}$

Multiply value of $[M]$ by $1 \times 10^{-6}$, so from $molecules \: m^{-3}$ to $molecules \: cm^{-3}$

Then for each timestep (row):
1. Create new, empty columns in dataset to store results of calculations
2. 
3. Calculate the source rate using $2 \cdot k[O^1D][H_2O]$

In [ ]:
## Unit conversions
from datetime import datetime, timezone
import numpy as np
from pytz import timezone

kb = 1.38e-23 # J K^−1 Boltzmann's constant

for i in range(len(df)):
    # Convert the dates to a timestamp value so that the difference between times can be calculated as an integer
    df.loc[i, "t"] = df["UTC_Start_dt"][i].replace().timestamp() - df["UTC_Start_dt"][0].replace().timestamp()
    df.loc[i, "M"] = ((df.loc[i, "Pres"] * 100) / (kb * df.loc[i, "Temp"])) * 1e-6 ## molecules cm^3 pressure was in hPa

df.head()


Calculate $[O^1D]$ using $[O^1D]_{SS} = \frac{J[O_3]}{k_1[H_2O] + k_2}$

(NOT $k_2[M]$ as this cancels out after the conversion to $molecules \: cm^{-3}$ for the other concentrations

Need to calculate $[O^1D]$ indirectly using data for $[O_3]$

Production

$O_3 + hv → O_2 + O(^1D)$

Loss

$O(^1D) + M → O(^3P) + M$

$O(^1D) + H_2O → 2OH$

Assumed to be in steady state - due to its short lifetime

So $\frac{d[O^1D]}{dt} = J[O_3] - k_1[O^1D][H_2O] - k_2[O^1D] = 0$

$[O^1D]_{SS} = \frac{J[O_3]}{k_1[H_2O] + k_2}$

$(M = \frac{p}{kT})$

$k = Ae^{-\frac{E_a}{RT}}$ to calculate temperature dependent rate constants

In [ ]:
df['kH2O'] = 1.63e-10 * np.exp(-60/df["Temp"]) ## page 21 of https://jpldataeval.jpl.nasa.gov/pdf/NASA-JPL%20Evaluation%2019-5.pdf
df['kM'] = 2.15e-11 * np.exp(-110/df["Temp"]) ## ## page 21 of https://jpldataeval.jpl.nasa.gov/pdf/NASA-JPL%20Evaluation%2019-5.pdf


df["O3"] = df["O3_M"] * df["M"]
df["H2O"] = df["H2O_M"] *  df["M"]
df["[OH]_measured"] = df["OH_ATHOS"] * 1e-12 * df["M"] # in ppt


In [ ]:
for i in range(len(df)):
    df.loc[i, "O1D"] = (df["jO3_O2_O1D_CAFS"][i] * df["O3_M"][i]) / ((df['kH2O'][i] * df["H2O_M"][i]) + (df['kM'][i]))
    
    # Photoloysis frequency, concentration of water etc on top of fraction
    df.loc[i, "source"] = 2 * df['kH2O'][i] * df["O1D"][i] * df["H2O"][i]
print(df["O1D"].mean())
print(df["source"].mean())
df.head()

# source should be 10e5

## Plot date vs source rate and compare with altitude 

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["source"][df["date"]< "2016-07-30"], label = "Date vs calculated source rate")
plt.xlabel("date")
plt.ylabel("source rate / molecules cm$^{-3} s^{-1}$")
plt.legend(loc='upper right')

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["Altitude"][df["date"]< "2016-07-30"], label = "Date vs altitude")
plt.xlabel("date")
plt.ylabel("altitude / m")
plt.legend(loc='upper right')

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["source"][df["date"]< "2016-07-30"], y = df["Altitude"][df["date"]< "2016-07-30"], label = "Date vs altitude")
plt.xlabel("source")
plt.ylabel("altitude / m")
plt.legend(loc='upper right')

The graph shows that the source rate in OH is higher at low altitudes and lower at high altitudes - we can link this to the source of OH being the reaction between $O(^1D)$ and $H_2O$. Therefore this varying rate must be related to one (or both) of these species.

## Plot altitude dependence of $H_2O$ and $O(^1D)$

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["H2O_M"][df["date"]< "2016-07-30"], label = "Date vs measured H2O")
plt.xlabel("date")
plt.ylabel("$[H_2O]$ / molecules cm$^{-3}$")
plt.legend(loc='upper right')

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["jO3_O2_O1D_CAFS"][df["date"]< "2016-07-30"], label = "Date vs calculated O(1D)")
plt.xlabel("date")
plt.ylabel("$[O(^1D)]$ / molecules cm$^{-3}$")
plt.legend(loc='upper right')

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["Altitude"][df["date"]< "2016-07-30"], label = "Date vs measured altitude")
plt.xlabel("date")
plt.ylabel("altitude / m")
plt.legend(loc='upper right')

## Local time
## Measured vs concentration - legend

From this graph it appears that there is a strong altitude dependence on concentrations of $H_2O$ in the atmosphere and the graph for $H_2O$ vs date is very similar to the graph for altitude vs date.

Seems that $H_2O$ has the bigger altitude dependence, whereas $O(^1D)$ seems to be affected more by the hour of day.

## Box model equation to find [OH]

## Steady state of OH

As $\frac{d[OH]}{dt} = Σsources - Σsinks$

$\frac{d[OH]}{dt} = P(OH) - k_1[OH][CO] - k_2[CH_4][OH]$

At steady state the sources and sinks balance:

$P(OH) - k_1[OH][CO] - k_2[CH_4][OH] = 0$

As we do not know [OH] (calculated) yet we cannot calculate the sink rate however we can use the steady state approximation to rearrange to find $[OH]$ from the other known values

$P(OH) = k_1[OH][CO] + k_2[CH_4][OH]$

$P(OH) = [OH](k_1[CO] + k_2[CH_4])$

$[OH] = \frac{p(OH)}{(k_1[CO] + k_2[CH4])}$

$[OH] = \frac{S}{(k_1[CO] + k_2[CH_4])}$

In [ ]:
df["source"].mean()

In [ ]:
df['kCO'] = 1.85e-13 * np.exp(-65/(df['Temp'])) ## page 484 of https://jpldataeval.jpl.nasa.gov/pdf/NASA-JPL%20Evaluation%2019-5.pdf
df['kCH4'] = 2.45e-12 * np.exp(-1775/df['Temp']) ## page 106 of https://jpldataeval.jpl.nasa.gov/pdf/NASA-JPL%20Evaluation%2019-5.pdf
df["CO"] = df['CO_M'] * df["M"]
df["CH4"] = df['CH4_M'] * df["M"]

for i in range(len(df)):
    df.loc[i, "[OH]_calc"] = (df.loc[i, "source"])/ ((df['kCO'][i] * df["CO"][i]) + (df['kCH4'][i] * df["CH4"][i]))
df["[OH]_calc"]

## Calculations - sinks

Sink = $k_1[OH][CO] + k_2[CH_4][OH]$

Use the _calculated_ value of $[OH]$ for this

In [ ]:
for i in range(len(df)):
    df.loc[i, "sink"] = (df['kCO'][i] * df["[OH]_calc"][i] * df['CO'][i]) + (df['kCH4'][i] * df["CH4"][i] * df["[OH]_calc"][i])

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["sink"][df["date"]< "2016-07-30"], label = "Date vs calculated sink rate")
plt.xlabel("date")
plt.ylabel("sink rate of OH / molecules cm$^{-3} s^{-1}$")
plt.legend(loc='upper right')

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"]< "2016-07-30"], y = df["Altitude"][df["date"]< "2016-07-30"], label = "Date vs measured altitude")
plt.xlabel("date")
plt.ylabel("altitude / m")
plt.legend(loc='upper right')

## $\frac{d[OH]}{dt} = P - L$ (production P is source rate, loss L is sink rate)

In [ ]:
## Checking values for d[OH]/dt are 0 (or as close to 0 as can be represented by a computer)
for i in range(len(df)):
    df.loc[i, "dOH/dt"] = (df['source'][i] - df['sink'][i])
df['dOH/dt']

## Plotting calculated $[OH]_{SS}$ vs measured $[OH]$ from ATom data

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"] < "2016-07-30"], y = df["[OH]_calc"][df["date"] < "2016-07-30"], label = "Calculated OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend()
# fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][df["date"] < "2016-07-30"], y = df["[OH]_measured"][df["date"] < "2016-07-30"], label = "Measured OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend()

import pickle
with open("comparison.pkl", "wb") as f:
    pickle.dump(fig, f)

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], y = df["[OH]_calc"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], label = "Calculated OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend()
# fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], y = df["[OH]_measured"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], label = "Measured OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend()


Comparison plots for 4 days

In [ ]:
plt.rcParams.update({'font.size':20})

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][(df["date"] > "2016-07-28") & (df["date"] < "2016-07-30")], y = df["[OH]_calc"][(df["date"] > "2016-07-28") & (df["date"] < "2016-07-30")], label = "Calculated OH")
plt.scatter(x = df["date"][(df["date"] > "2016-07-28") & (df["date"] < "2016-07-30")], y = df["[OH]_measured"][(df["date"] > "2016-07-28") & (df["date"] < "2016-07-30")], label = "Measured OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend(prop={'size':20})

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][(df["date"] > "2016-07-30") & (df["date"] < "2016-08-02")], y = df["[OH]_calc"][(df["date"] > "2016-07-30") & (df["date"] < "2016-08-02")], label = "Calculated OH")
plt.scatter(x = df["date"][(df["date"] > "2016-07-30") & (df["date"] < "2016-08-02")], y = df["[OH]_measured"][(df["date"] > "2016-07-30") & (df["date"] < "2016-08-02")], label = "Measured OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend(prop={'size':20})

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][(df["date"] > "2016-08-02") & (df["date"] < "2016-08-04")], y = df["[OH]_calc"][(df["date"] > "2016-08-02") & (df["date"] < "2016-08-04")], label = "Calculated OH")
plt.scatter(x = df["date"][(df["date"] > "2016-08-02") & (df["date"] < "2016-08-04")], y = df["[OH]_measured"][(df["date"] > "2016-08-02") & (df["date"] < "2016-08-04")], label = "Measured OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend(prop={'size':20})

fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(x = df["date"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], y = df["[OH]_calc"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], label = "Calculated OH")
plt.scatter(x = df["date"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], y = df["[OH]_measured"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")], label = "Measured OH")
plt.xlabel("date")
plt.ylabel("[OH] / molecules cm$^{-3}$")
plt.legend(prop={'size':20})

$r^2$ scores for 4 days

In [ ]:
from sklearn.linear_model import LinearRegression
import sklearn
X = df[["[OH]_measured"]][(df["date"] > "2016-07-28") & (df["date"] < "2016-07-30")]
y = df["[OH]_calc"][(df["date"] > "2016-07-28") & (df["date"] < "2016-08-01")]
print(sklearn.metrics.r2_score(X, y))

X = df[["[OH]_measured"]][(df["date"] > "2016-07-30") & (df["date"] < "2016-08-02")]
y = df["[OH]_calc"][(df["date"] > "2016-07-30") & (df["date"] < "2016-08-02")]
print(sklearn.metrics.r2_score(X, y))

X = df[["[OH]_measured"]][(df["date"] > "2016-08-02") & (df["date"] < "2016-08-04")]
y = df["[OH]_calc"][(df["date"] > "2016-08-02") & (df["date"] < "2016-08-04")]
print(sklearn.metrics.r2_score(X, y))

X = df[["[OH]_measured"]][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")]
y = df["[OH]_calc"][(df["date"] > "2016-08-05") & (df["date"] < "2016-08-07")]
print(sklearn.metrics.r2_score(X, y))

X = df[["[OH]_measured"]]
y = df["[OH]_calc"]
print(sklearn.metrics.r2_score(X, y))

## Some exploratory comparison plots

### Ratio of modelled to measured

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
ratio = df["[OH]_calc"]/df["[OH]_measured"]
plt.scatter(x = df["date"][df["date"] < "2016-07-30"], y = ratio[df["date"] < "2016-07-30"])
plt.xlabel("date")
plt.ylabel("ratio of modelled [OH] / observed [OH]")

Fluctuation in modelled values is reflected here.

### Percentage difference

Using $PD = 100 \times \frac{(observation - model)}{(observation + model) / 2}$

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
pd = 100 * ((df["[OH]_measured"] - df["[OH]_calc"]) / ((df["[OH]_measured"] + df["[OH]_calc"]) / 2))
plt.scatter(x = df["date"][df["date"] < "2016-07-30"], y = pd[df["date"] < "2016-07-30"], label = "Calculated OH")
plt.xlabel("date")
plt.ylabel("percent difference / %")
plt.legend()

Again, the fluctuation causes quite a big percentage difference at both ends of the y axis

### Scatterplot of modelled vs measured

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][(df["date"] < "2016-07-30") & (df["Altitude"] < 3000)], y = df["[OH]_measured"][(df["date"] < "2016-07-30") & (df["Altitude"] < 3000)], label = "altitude < 3000 m")
# fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][(df["date"] < "2016-07-30") & (df["Altitude"] >= 3000) & (df["Altitude"] < 6000)], y = df["[OH]_measured"][(df["date"] < "2016-07-30") & (df["Altitude"] >= 3000) & (df["Altitude"] < 6000)], label = "altitude >= 3000 m, < 6000 m")
# fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][(df["date"] < "2016-07-30") & (df["Altitude"] >= 6000)], y = df["[OH]_measured"][(df["date"] < "2016-07-30") & (df["Altitude"] >= 6000)], label = "altitude >= 6000 m")
plt.xlabel("modelled [OH] / molecules cm$^{-3}$")
plt.ylabel("measured [OH] / molecules cm$^{-3}$")
ax.axline([0,0], [100,100], label = "1:1 ratio line")

slope, intercept = np.polyfit(df["[OH]_calc"], df["[OH]_measured"], 1)
x = df["[OH]_calc"]
y = df["[OH]_measured"]
plt.plot(x, slope * x + intercept, color='red', label=f'Best fit line: slope = {round(slope, 2)}')
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][ (df["Altitude"] < 3000)], y = df["[OH]_measured"][(df["Altitude"] < 3000)], label = "altitude < 3000 m")
# fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][(df["Altitude"] >= 3000) & (df["Altitude"] < 6000)], y = df["[OH]_measured"][(df["Altitude"] >= 3000) & (df["Altitude"] < 6000)], label = "altitude >= 3000 m, < 6000 m")
# fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][(df["Altitude"] >= 6000)], y = df["[OH]_measured"][(df["Altitude"] >= 6000)], label = "altitude >= 6000 m")
plt.xlabel("modelled [OH] / molecules cm$^{-3}$")
plt.ylabel("measured [OH] / molecules cm$^{-3}$")
ax.axline([0,0], [100,100], label = "1:1 ratio line")

slope, intercept = np.polyfit(df["[OH]_calc"], df["[OH]_measured"], 1)
x = df["[OH]_calc"]
y = df["[OH]_measured"]
plt.plot(x, slope * x + intercept, color='red', label=f'Best fit line: slope = {round(slope, 2)}')
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][ (df["Altitude"] < 3000)], y = df["[OH]_measured"][(df["Altitude"] < 3000)], label = "altitude < 3000 m")
plt.title("altitude < 3000 m")
plt.xlabel("modelled [OH] / molecules cm$^{-3}$")
plt.ylabel("measured [OH] / molecules cm$^{-3}$")
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][(df["Altitude"] >= 3000) & (df["Altitude"] < 6000)], y = df["[OH]_measured"][(df["Altitude"] >= 3000) & (df["Altitude"] < 6000)], label = "altitude >= 3000 m, < 6000 m")
plt.title("altitude >= 3000 m, < 6000 m")
plt.xlabel("modelled [OH] / molecules cm$^{-3}$")
plt.ylabel("measured [OH] / molecules cm$^{-3}$")
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["[OH]_calc"][(df["Altitude"] >= 6000)], y = df["[OH]_measured"][(df["Altitude"] >= 6000)], label = "altitude >= 6000 m")
plt.xlabel("modelled [OH] / molecules cm$^{-3}$")
plt.ylabel("measured [OH] / molecules cm$^{-3}$")
plt.title("altitude >= 6000 m")
plt.xlabel("modelled [OH] / molecules cm$^{-3}$")
plt.ylabel("measured [OH] / molecules cm$^{-3}$")
# ax.axline([0,0], [100,100])
plt.legend()

Can see a relatively positive correlation between modelled and measured values

In [ ]:
df

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(df["[OH]_measured"], df["[OH]_calc"])

In [ ]:
df["ratio"] = (df["[OH]_calc"]/df["[OH]_measured"])
# df["ratio"].mean(axis=1)
df.mean()

## Machine learning experimentation

### Resampling - may be useful when doing less random train-test-splitting

In [ ]:
df_resampled = df.resample('H', on = 'date').mean()
df_resampled

In [ ]:
(df_resampled.index)

In [ ]:
plt.plot(df_resampled.index, df_resampled["[OH]_calc"])

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

In [ ]:
df_inputs = df[['Temp', 'Pres', 'jO3_O2_O1D_CAFS', 'O3_M', 'H2O_M', 'CO_M', 'CH4_M']]

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# Example data
X = df_inputs   # 100 samples, 5 input features
y = df["[OH]_calc"]  # single output

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# GBRT model
model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)


In [ ]:
from sklearn.linear_model import LinearRegression
X = df_inputs
y = df["[OH]_calc"]
reg = LinearRegression().fit(X, y)
out_calc = reg.predict(X)
reg.score(X, y)


In [ ]:
from sklearn.linear_model import LinearRegression
X = df_inputs
y = df["[OH]_measured"]
reg = LinearRegression().fit(X, y)
out_measured = reg.predict(X)
reg.score(X, y)

In [ ]:
fig, ax = plt.subplots(figsize = (24, 6))
plt.scatter(df["date"][(df["date"] < "2016-07-30")], df["[OH]_calc"][(df["date"] < "2016-07-30")], label = "calc")
plt.scatter(df["date"][(df["date"] < "2016-07-30")], df["[OH]_measured"][(df["date"] < "2016-07-30")], label = "measured")
plt.scatter(df["date"][(df["date"] < "2016-07-30")], out_calc[(df["date"] < "2016-07-30")], label = "pred calc")
plt.scatter(df["date"][(df["date"] < "2016-07-30")], out_measured[(df["date"] < "2016-07-30")], label = "pred measured")
plt.legend()

In [ ]:
df_inputs.shape

In [ ]:
df_inputs.columns

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X = df_inputs # 100 samples, 5 input features
y = df["[OH]_calc"]  # single output

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

model = LinearRegression()

model.fit(X_train, y_train)

importance = model.coef_

for i, v in enumerate(importance):
    print("Feature: %0d, Score: %.5f" % (i, v))

plt.bar([x for x in range(len(importance))], importance)
plt.show()    

In [ ]:
df_inputs.columns

In [ ]:
from sklearn.ensemble import RandomForestRegressor


X = df_inputs   # 100 samples, 5 input features
y = df["[OH]_calc"]  # single output

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42
)

model = RandomForestRegressor()


model.fit(X_train, y_train)

print(f"model score on training data: {model.score(X_train, y_train)}")
print(f"model score on testing data: {model.score(X_test, y_test)}")

In [ ]:
importances = model.feature_importances_

In [ ]:
indices = np.argsort(importances)

fig, ax = plt.subplots()
ax.barh(range(len(importances)), importances[indices])
ax.set_yticks(range(len(importances)))
_ = ax.set_yticklabels(np.array(X_train.columns)[indices])

In [ ]:
import sklearn
from sklearn.linear_model import Ridge

X = df_inputs   # 100 samples, 5 input features
y = df["[OH]_calc"]  # single output

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=1
)

ridgeReg = Ridge(alpha = 1)
ridgeReg.fit(X_train, y_train)

print(ridgeReg.score(X_train, y_train))
print(ridgeReg.score(X_test, y_test))

X = df_inputs   # 100 samples, 5 input features
y = df["[OH]_measured"]  # single output

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=1
)

ridgeReg = Ridge(alpha = 1)
ridgeReg.fit(X_train, y_train)

print(ridgeReg.score(X_train, y_train))
print(ridgeReg.score(X_test, y_test))